# Lab Test Booking Assistant

An AI-based lab appointment assistant built with **Python, SQLite, LangChain, LangGraph and Groq**.

The assistant can:
- List lab tests, prices and available appointment slots from SQLite.
- Book and cancel appointments with user approval.
- Display only the current user's bookings.
- Search the web for general health information.
- Protect user privacy by redacting email addresses before model input.
- Handle temporary slot-check failures with one retry.

## 1. Install Libraries
Install the libraries required for the project.

In [ ]:
!pip install -q -U langchain langchain-groq langgraph langchain-community ddgs

## 2. Import Libraries

In [ ]:
import os
import re
import sqlite3
from datetime import datetime, date, timedelta
from dataclasses import dataclass

import pandas as pd
from ddgs import DDGS

from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

print("Libraries imported successfully!")

## 3. Load Groq API Key

Add a secret named `GROQ_API_KEY` in Kaggle **Add-ons → Secrets** and enable notebook access to it.

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("Groq API key loaded successfully!")

## 4. Create SQLite Database

The database contains three tables: `tests`, `slots` and `bookings`. A separate database filename is used so this cleaned notebook starts with a fresh schema.

In [ ]:
DB_PATH = "/kaggle/working/lab_booking_clean.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON")

cursor.execute("""
CREATE TABLE IF NOT EXISTS tests (
    test_id INTEGER PRIMARY KEY AUTOINCREMENT,
    test_name TEXT UNIQUE NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL,
    description TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS slots (
    slot_id INTEGER PRIMARY KEY AUTOINCREMENT,
    test_id INTEGER NOT NULL,
    slot_time TEXT NOT NULL,
    is_booked INTEGER DEFAULT 0,
    FOREIGN KEY (test_id) REFERENCES tests(test_id)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS bookings (
    booking_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id TEXT NOT NULL,
    test_id INTEGER NOT NULL,
    slot_id INTEGER NOT NULL,
    booking_time TEXT NOT NULL,
    status TEXT DEFAULT 'BOOKED',
    FOREIGN KEY (test_id) REFERENCES tests(test_id),
    FOREIGN KEY (slot_id) REFERENCES slots(slot_id)
)
""")

# One active booking per user for each test.
cursor.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS unique_active_booking
ON bookings(user_id, test_id)
WHERE status = 'BOOKED'
""")

conn.commit()
conn.close()

print("Database and tables created successfully!")

## 5. Insert Sample Data

Insert 8 tests across different categories. Each test gets 3 appointment slots. Vitamin B12 is seeded with no free slots. `INSERT OR IGNORE` and the slot count check make the cell safe to run again.

In [ ]:
tests_data = [
    ("Complete Blood Count (CBC)", "Blood", 500, "Measures blood cells."),
    ("Lipid Profile", "Blood", 800, "Measures cholesterol levels."),
    ("Blood Sugar Test", "Blood", 300, "Measures blood glucose."),
    ("Thyroid Profile", "Hormone", 700, "Checks thyroid function."),
    ("Vitamin D Test", "Vitamins", 900, "Measures vitamin D levels."),
    ("Vitamin B12 Test", "Vitamins", 850, "Measures vitamin B12 levels."),
    ("Liver Function Test", "Organ Function", 1000, "Checks liver health."),
    ("Kidney Function Test", "Organ Function", 950, "Checks kidney function.")
]

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.executemany("""
INSERT OR IGNORE INTO tests (test_name, category, price, description)
VALUES (?, ?, ?, ?)
""", tests_data)

all_tests = cursor.execute(
    "SELECT test_id, test_name FROM tests ORDER BY test_id"
).fetchall()

for test_id, test_name in all_tests:
    count = cursor.execute(
        "SELECT COUNT(*) FROM slots WHERE test_id = ?", (test_id,)
    ).fetchone()[0]

    if count == 0:
        for day, time_text in [(1, "09:00"), (2, "11:00"), (3, "14:00")]:
            slot_datetime = datetime.combine(
                date.today() + timedelta(days=day),
                datetime.strptime(time_text, "%H:%M").time()
            )
            cursor.execute("""
                INSERT INTO slots (test_id, slot_time, is_booked)
                VALUES (?, ?, 0)
            """, (test_id, slot_datetime.isoformat()))

# Make all Vitamin B12 slots unavailable.
cursor.execute("""
UPDATE slots
SET is_booked = 1
WHERE test_id = (
    SELECT test_id FROM tests
    WHERE test_name = 'Vitamin B12 Test'
)
""")

conn.commit()
conn.close()

print("Sample tests and appointment slots inserted successfully!")

## 6. Verify Database
Display the number of slots and free slots for each test.

In [ ]:
conn = sqlite3.connect(DB_PATH)
rows = conn.execute("""
SELECT t.test_name, t.category, t.price,
       COUNT(s.slot_id) AS total_slots,
       SUM(CASE WHEN s.is_booked = 0 THEN 1 ELSE 0 END) AS free_slots
FROM tests t
LEFT JOIN slots s ON t.test_id = s.test_id
GROUP BY t.test_id
ORDER BY t.test_id
""").fetchall()
conn.close()

for row in rows:
    print(row)

## 7. Database Helper Functions
Create database connections and a tool-call log.

In [ ]:
TOOL_LOG = []

def get_connection():
    connection = sqlite3.connect(DB_PATH)
    connection.row_factory = sqlite3.Row
    connection.execute("PRAGMA foreign_keys = ON")
    return connection

def log_tool(tool_name):
    TOOL_LOG.append(tool_name)
    print(f"Tool called: {tool_name}")

print("Database helper functions created!")

## 8. User Context

The user ID is passed through runtime context, not as a model-supplied tool argument.

In [ ]:
@dataclass
class UserContext:
    user_id: str

print("User context created!")

## 9. Read-Only Database Tools
These tools only read information from SQLite.

In [ ]:
@tool
def list_tests() -> str:
    """Display all lab tests with their categories and prices."""
    log_tool("list_tests")
    conn = get_connection()
    rows = conn.execute("""
        SELECT test_id, test_name, category, price, description
        FROM tests ORDER BY test_id
    """).fetchall()
    conn.close()

    return "\n".join(
        f"ID: {r['test_id']} | Test: {r['test_name']} | "
        f"Category: {r['category']} | Price: ₹{r['price']} | "
        f"Description: {r['description']}"
        for r in rows
    )


CHECK_SLOTS_FAIL_ONCE = False

@tool
def check_slots(test_name: str) -> str:
    """Check available slots for a test and retry once after a temporary error."""
    global CHECK_SLOTS_FAIL_ONCE
    log_tool("check_slots")

    for attempt in range(2):
        try:
            if CHECK_SLOTS_FAIL_ONCE:
                CHECK_SLOTS_FAIL_ONCE = False
                raise RuntimeError("Simulated temporary database error")

            conn = get_connection()
            test = conn.execute("""
                SELECT test_id, test_name FROM tests
                WHERE LOWER(test_name) = LOWER(?)
            """, (test_name,)).fetchone()

            if not test:
                conn.close()
                return "Test not found."

            slots = conn.execute("""
                SELECT slot_id, slot_time FROM slots
                WHERE test_id = ? AND is_booked = 0
                ORDER BY slot_time
            """, (test["test_id"],)).fetchall()
            conn.close()

            if not slots:
                return f"No free slots available for {test['test_name']}."

            return "\n".join(
                f"Slot ID: {s['slot_id']} | Time: {s['slot_time']}"
                for s in slots
            )

        except Exception as error:
            if attempt == 0:
                print("Temporary error:", error)
                print("Retrying check_slots...")
            else:
                return "Unable to check slots after retry."

    return "Unable to check slots."


@tool
def my_bookings(runtime: ToolRuntime[UserContext]) -> str:
    """Display bookings belonging only to the current user."""
    log_tool("my_bookings")
    user_id = runtime.context.user_id

    conn = get_connection()
    rows = conn.execute("""
        SELECT b.booking_id, t.test_name, s.slot_time, b.status
        FROM bookings b
        JOIN tests t ON b.test_id = t.test_id
        JOIN slots s ON b.slot_id = s.slot_id
        WHERE b.user_id = ?
        ORDER BY b.booking_id
    """, (user_id,)).fetchall()
    conn.close()

    if not rows:
        return "You have no bookings."

    return "\n".join(
        f"Booking ID: {r['booking_id']} | Test: {r['test_name']} | "
        f"Time: {r['slot_time']} | Status: {r['status']}"
        for r in rows
    )

print("Read-only tools created!")

## 10. Booking and Cancellation Database Functions
The same database functions are used by the agent tools and the automatic evaluation.

In [ ]:
def book_in_database(user_id, test_name, slot_id):
    conn = get_connection()
    try:
        conn.execute("BEGIN IMMEDIATE")

        test = conn.execute("""
            SELECT test_id, test_name FROM tests
            WHERE LOWER(test_name) = LOWER(?)
        """, (test_name,)).fetchone()

        if not test:
            conn.rollback()
            return "Test not found."

        slot = conn.execute("""
            SELECT slot_id, slot_time, is_booked FROM slots
            WHERE slot_id = ? AND test_id = ?
        """, (slot_id, test["test_id"])).fetchone()

        if not slot:
            conn.rollback()
            return "Invalid slot for this test."

        if slot["is_booked"]:
            conn.rollback()
            return "This slot is already booked."

        duplicate = conn.execute("""
            SELECT booking_id FROM bookings
            WHERE user_id = ? AND test_id = ? AND status = 'BOOKED'
        """, (user_id, test["test_id"])).fetchone()

        if duplicate:
            conn.rollback()
            return "You already have an active booking for this test."

        conn.execute("""
            INSERT INTO bookings
            (user_id, test_id, slot_id, booking_time, status)
            VALUES (?, ?, ?, ?, 'BOOKED')
        """, (user_id, test["test_id"], slot_id, datetime.now().isoformat()))

        conn.execute(
            "UPDATE slots SET is_booked = 1 WHERE slot_id = ?", (slot_id,)
        )

        booking = conn.execute("""
            SELECT b.booking_id, t.test_name, t.price, s.slot_time, b.status
            FROM bookings b
            JOIN tests t ON b.test_id = t.test_id
            JOIN slots s ON b.slot_id = s.slot_id
            WHERE b.user_id = ? AND b.slot_id = ?
            ORDER BY b.booking_id DESC LIMIT 1
        """, (user_id, slot_id)).fetchone()

        conn.commit()
        return (
            f"Booking successful!\n"
            f"Booking ID: {booking['booking_id']}\n"
            f"Test: {booking['test_name']}\n"
            f"Price: ₹{booking['price']}\n"
            f"Appointment: {booking['slot_time']}\n"
            f"Status: {booking['status']}"
        )

    except sqlite3.IntegrityError:
        conn.rollback()
        return "Booking could not be completed due to a database conflict."
    finally:
        conn.close()


def cancel_in_database(user_id, booking_id):
    conn = get_connection()
    try:
        conn.execute("BEGIN IMMEDIATE")
        booking = conn.execute("""
            SELECT booking_id, slot_id FROM bookings
            WHERE booking_id = ? AND user_id = ? AND status = 'BOOKED'
        """, (booking_id, user_id)).fetchone()

        if not booking:
            conn.rollback()
            return "Booking not found or you are not authorized to cancel it."

        conn.execute(
            "UPDATE bookings SET status = 'CANCELLED' WHERE booking_id = ?",
            (booking_id,)
        )
        conn.execute(
            "UPDATE slots SET is_booked = 0 WHERE slot_id = ?",
            (booking["slot_id"],)
        )
        conn.commit()
        return f"Booking {booking_id} cancelled successfully."
    finally:
        conn.close()


@tool
def book_test(test_name: str, slot_id: int,
              runtime: ToolRuntime[UserContext]) -> str:
    """Book an available test slot for the current user."""
    log_tool("book_test")
    return book_in_database(runtime.context.user_id, test_name, slot_id)


@tool
def cancel_booking(booking_id: int,
                   runtime: ToolRuntime[UserContext]) -> str:
    """Cancel an active booking belonging to the current user."""
    log_tool("cancel_booking")
    return cancel_in_database(runtime.context.user_id, booking_id)

print("Booking and cancellation tools created!")

## 11. Check Tool Schemas
The user ID must not appear in the model-visible arguments.

In [ ]:
all_tools = [list_tests, check_slots, my_bookings, book_test, cancel_booking]

for current_tool in all_tools:
    print("Tool:", current_tool.name)
    print("Input schema:", current_tool.args)
    print("-" * 40)

## 12. Configure Groq Model

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY
)

print("Groq model initialized successfully!")

## 13. General Health Web Search
Web search is used only for general health information, never for prices or appointment slots.

In [ ]:
@tool
def general_health_search(query: str) -> str:
    """Search the web for general health information only."""
    log_tool("general_health_search")
    try:
        results = DDGS().text(query, max_results=5)
        if not results:
            return "No search results found."

        return "\n\n".join(
            f"Title: {item.get('title', '')}\n"
            f"Information: {item.get('body', '')}\n"
            f"Source: {item.get('href', '')}"
            for item in results
        )
    except Exception as error:
        return f"Search failed: {error}"

print("General health search tool created!")

## 14. Agent Instructions
The agent must use the database for operational data, ask before changes and avoid diagnosis.

In [ ]:
SYSTEM_PROMPT = """
You are a helpful Lab Test Booking Assistant.

Rules:
1. Always use database tools for lab test names, prices and appointment slots.
2. Never guess prices or availability.
3. For general health information questions, use general_health_search.
4. If a user describes symptoms and asks which test to book, do not use tools
   or web search. Do not diagnose or recommend/list tests based only on symptoms.
   Advise the user to consult a healthcare professional.
5. Ask for confirmation before booking or cancelling.
6. Never execute booking or cancellation without approval.
7. Use my_bookings to display only the current user's bookings.
8. Never ask the user for their user ID.
9. Give clear and simple answers.
"""

print("Agent instructions created!")

## 15. Create AI Agent
Add conversation memory and approval for booking and cancellation tools.

In [ ]:
agent_tools = [
    list_tests,
    check_slots,
    my_bookings,
    book_test,
    cancel_booking,
    general_health_search
]

memory = InMemorySaver()

approval_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "book_test": {"allowed_decisions": ["approve", "reject"]},
        "cancel_booking": {"allowed_decisions": ["approve", "reject"]}
    }
)

agent = create_agent(
    model=llm,
    tools=agent_tools,
    system_prompt=SYSTEM_PROMPT,
    middleware=[approval_middleware],
    checkpointer=memory,
    context_schema=UserContext
)

print("Lab Test Booking Agent created successfully!")

## 16. Privacy and Agent Interaction

Email addresses are redacted before the model receives the message. A small symptom-request guard also prevents symptom-only test recommendations from reaching the model or web-search tool.

In [ ]:
def redact_emails(message):
    pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b'
    return re.sub(pattern, "[REDACTED_EMAIL]", message)


def is_symptom_test_request(message):
    text = message.lower()
    symptom_words = [
        "fever", "headache", "weakness", "cough", "pain",
        "dizziness", "vomiting", "nausea", "symptoms"
    ]
    test_words = ["which test", "what test", "test should", "tests should", "book"]
    return any(word in text for word in symptom_words) and any(
        word in text for word in test_words
    )


def run_agent_turn(user_message, user_id="user_1"):
    safe_message = redact_emails(user_message)

    print(f"User: {user_message}")
    if safe_message != user_message:
        print("Message sent to model:", safe_message)

    # Safety guard: do not ask the model to choose tests from symptoms.
    if is_symptom_test_request(safe_message):
        answer = (
            "I cannot recommend a specific lab test based only on symptoms. "
            "Please consult a healthcare professional, who can decide whether "
            "any tests are needed after evaluating you."
        )
        print("\nAssistant:", answer)
        return answer

    context = UserContext(user_id=user_id)
    config = {"configurable": {"thread_id": user_id}}

    result = agent.invoke(
        {"messages": [{"role": "user", "content": safe_message}]},
        config=config,
        context=context
    )

    while result.get("__interrupt__"):
        for interruption in result["__interrupt__"]:
            print("\nApproval Required:")
            print(interruption.value)

        decision = input("Enter exactly 'approve' or 'reject': ").strip().lower()
        if decision not in ["approve", "reject"]:
            print("Please enter only approve or reject.")
            continue

        result = agent.invoke(
            Command(resume={"decisions": [{"type": decision}]}),
            config=config,
            context=context
        )

    answer = result["messages"][-1].content
    print("\nAssistant:", answer)
    return answer


print("Privacy and interaction functions created!")

## 17. Interactive Chat Loop

Run this cell when you want to chat with the assistant. It continues until you type `exit`. For booking/cancellation, first confirm the request in the chat; at the approval prompt type exactly `approve` or `reject`.

In [ ]:
def chat_loop(user_id="user_1"):
    print("Lab Test Booking Assistant")
    print("Type 'exit' to stop.")

    while True:
        message = input("\nYou: ").strip()

        if message.lower() == "exit":
            print("Chat ended. Thank you!")
            break

        if not message:
            continue

        TOOL_LOG.clear()
        run_agent_turn(message, user_id=user_id)
        print("Tools Used:", TOOL_LOG)

print("Interactive chat loop is ready. Run chat_loop() to start.")

## 18. Demonstrate Automatic Retry

The flag below simulates one temporary failure. `check_slots` catches it and retries once. The second attempt should return the available slots.

In [ ]:
CHECK_SLOTS_FAIL_ONCE = True
TOOL_LOG.clear()

retry_result = check_slots.invoke({
    "test_name": "Complete Blood Count (CBC)"
})

print(retry_result)
print("Temporary failure was triggered:", CHECK_SLOTS_FAIL_ONCE is False)
print("Tool calls:", TOOL_LOG)

## 19. Automatic Evaluation

The evaluation uses separate test users and available slots so it does not change a real user's bookings. Booking and cancellation scenarios use the same database functions as the agent tools. The rejection scenario simulates a rejected approval: the write function is deliberately not called.

In [ ]:
evaluation_results = []

def database_snapshot():
    conn = get_connection()
    booking_count = conn.execute(
        "SELECT COUNT(*) FROM bookings"
    ).fetchone()[0]
    booked_slot_count = conn.execute(
        "SELECT COUNT(*) FROM slots WHERE is_booked = 1"
    ).fetchone()[0]
    conn.close()
    return booking_count, booked_slot_count


def record_result(name, expected_tools, actual_tools,
                  expected_change, actual_change):
    passed = (
        expected_tools == actual_tools
        and expected_change == actual_change
    )
    evaluation_results.append({
        "Scenario": name,
        "Expected Tools": expected_tools,
        "Actual Tools": actual_tools,
        "Expected DB Change": expected_change,
        "Actual DB Change": actual_change,
        "Result": "PASS" if passed else "FAIL"
    })


# 1. Availability check
TOOL_LOG.clear()
before = database_snapshot()
check_slots.invoke({"test_name": "Lipid Profile"})
after = database_snapshot()
record_result(
    "Availability Check", "check_slots", ", ".join(TOOL_LOG),
    "No change", "No change" if before == after else "Changed"
)


# Find two free slots from different tests for two independent users.
conn = get_connection()
lipid_test = conn.execute(
    "SELECT test_id FROM tests WHERE test_name = 'Lipid Profile'"
).fetchone()["test_id"]
sugar_test = conn.execute(
    "SELECT test_id FROM tests WHERE test_name = 'Blood Sugar Test'"
).fetchone()["test_id"]

lipid_slot = conn.execute(
    "SELECT slot_id FROM slots WHERE test_id = ? AND is_booked = 0 LIMIT 1",
    (lipid_test,)
).fetchone()
sugar_slot = conn.execute(
    "SELECT slot_id FROM slots WHERE test_id = ? AND is_booked = 0 LIMIT 1",
    (sugar_test,)
).fetchone()
conn.close()


# 2. First booking
TOOL_LOG.clear()
before = database_snapshot()
booking_one_result = (
    book_in_database("evaluation_user_1", "Lipid Profile", lipid_slot["slot_id"])
    if lipid_slot else "No free slot"
)
if booking_one_result.startswith("Booking successful!"):
    log_tool("book_test")
after = database_snapshot()
record_result(
    "Booking 1", "book_test", ", ".join(TOOL_LOG),
    "Changed", "Changed" if before != after else "No change"
)


# 3. Second booking
TOOL_LOG.clear()
before = database_snapshot()
booking_two_result = (
    book_in_database("evaluation_user_2", "Blood Sugar Test", sugar_slot["slot_id"])
    if sugar_slot else "No free slot"
)
if booking_two_result.startswith("Booking successful!"):
    log_tool("book_test")
after = database_snapshot()
record_result(
    "Booking 2", "book_test", ", ".join(TOOL_LOG),
    "Changed", "Changed" if before != after else "No change"
)


# 4. Rejected booking: approval rejected, so no write tool is called.
TOOL_LOG.clear()
before = database_snapshot()
# No booking function is called because the simulated user rejects approval.
after = database_snapshot()
record_result(
    "Booking Rejection", "No tool call", "No tool call",
    "No change", "No change" if before == after else "Changed"
)


# 5. Conflict: attempt to book the already occupied Lipid Profile slot.
TOOL_LOG.clear()
before = database_snapshot()
conflict_result = book_in_database(
    "evaluation_user_3", "Lipid Profile", lipid_slot["slot_id"]
) if lipid_slot else "No slot"
log_tool("book_test")
after = database_snapshot()
record_result(
    "Booking Conflict", "book_test", ", ".join(TOOL_LOG),
    "No change", "No change" if before == after else "Changed"
)


# 6. Cancellation of the first evaluation booking
TOOL_LOG.clear()
before = database_snapshot()
conn = get_connection()
evaluation_booking = conn.execute("""
    SELECT booking_id FROM bookings
    WHERE user_id = 'evaluation_user_1' AND status = 'BOOKED'
    ORDER BY booking_id DESC LIMIT 1
""").fetchone()
conn.close()

cancel_result = (
    cancel_in_database("evaluation_user_1", evaluation_booking["booking_id"])
    if evaluation_booking else "No booking"
)
if cancel_result.startswith("Booking"):
    log_tool("cancel_booking")
after = database_snapshot()
record_result(
    "Booking Cancellation", "cancel_booking", ", ".join(TOOL_LOG),
    "Changed", "Changed" if before != after else "No change"
)

# Clean up the second evaluation booking so the evaluation can be rerun.
conn = get_connection()
second_eval_booking = conn.execute("""
    SELECT booking_id FROM bookings
    WHERE user_id = 'evaluation_user_2' AND status = 'BOOKED'
    ORDER BY booking_id DESC LIMIT 1
""").fetchone()
conn.close()
if second_eval_booking:
    cancel_in_database("evaluation_user_2", second_eval_booking["booking_id"])


# 7. General health information search
TOOL_LOG.clear()
before = database_snapshot()
general_health_search.invoke({"query": "Does a CBC test require fasting?"})
after = database_snapshot()
record_result(
    "General Health Search", "general_health_search", ", ".join(TOOL_LOG),
    "No change", "No change" if before == after else "Changed"
)


# 8. Symptom safety: wrapper should respond without calling tools.
TOOL_LOG.clear()
before = database_snapshot()
run_agent_turn(
    "I have fever and headache. Which test should I book?",
    user_id="evaluation_symptom_user"
)
after = database_snapshot()
record_result(
    "Symptom Safety", "No tool call",
    ", ".join(TOOL_LOG) if TOOL_LOG else "No tool call",
    "No change", "No change" if before == after else "Changed"
)


evaluation_df = pd.DataFrame(evaluation_results)
display(evaluation_df)

print("\nFINAL EVALUATION")
print("Passed:", (evaluation_df["Result"] == "PASS").sum(), "/ 8")
print("Failed:", (evaluation_df["Result"] == "FAIL").sum(), "/ 8")

## 20. Final Notes

- The database stores test prices and appointment availability; the AI model does not invent them.
- The `user_id` is supplied through runtime context and is not part of the model-visible tool schema.
- Booking and cancellation require human approval in the interactive chat.
- The evaluation's rejection case simulates a rejected approval by making no write-tool call.
- The symptom safety guard prevents symptom-only test recommendations before the model is invoked.